# Qwen3 Layer Export - Export Model with Layer Hidden States

This notebook exports the Qwen3-4B model with exposed layer outputs for EVA-Ai FCP system.

**What it does:**
- Loads `RefalMachine/RuadaptQwen3-4B-Instruct` model
- Wraps it to capture `hidden_states` after each layer (36 layers)
- Exports to ONNX/OpenVINO format

**Output files:**
- `qwen_layer_model.pt` - PyTorch model with layer hooks
- `qwen_with_layers.onnx` - ONNX model (if export succeeds)
- `qwen_with_layers.xml/.bin` - OpenVINO model

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Output directory
OUTPUT_DIR = '/content/drive/MyDrive/EVA_Ai_Exports'
!mkdir -p $OUTPUT_DIR

In [ ]:
# Install dependencies
!pip install -q transformers torch openvino openvino-genai onnx

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('export')

In [ ]:
# Model configuration
MODEL_PATH = 'RefalMachine/RuadaptQwen3-4B-Instruct'
NUM_LAYERS = 36
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

logger.info(f"Using device: {DEVICE}")

In [ ]:
# Load model
logger.info(f"Loading model from {MODEL_PATH}")

config = AutoConfig.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)
logger.info(f"  hidden_size: {config.hidden_size}")
logger.info(f"  num_layers: {config.num_hidden_layers}")
logger.info(f"  num_heads: {config.num_attention_heads}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    config=config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE,
    low_cpu_mem_usage=True,
    output_hidden_states=True
)

model.eval()
logger.info(f"Model loaded successfully")

In [ ]:
# Model wrapper with layer hooks

class ModelWithLayerOutputs(torch.nn.Module):
    """
    Wraps Qwen model to return hidden_states after each layer.
    """
    def __init__(self, base_model, num_layers):
        super().__init__()
        self.base_model = base_model
        self.num_layers = num_layers
        self.layer_outputs = {}
        
        # Register hooks for all layers
        for idx, layer in enumerate(base_model.model.layers):
            layer.register_forward_hook(self._hook_fn(idx))

    def _hook_fn(self, layer_idx):
        def hook(module, input, output):
            self.layer_outputs[layer_idx] = output[0].detach()
        return hook

    def forward(self, input_ids, attention_mask=None):
        self.layer_outputs.clear()
        
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        # Collect all layer outputs
        all_outputs = {}
        
        # Embedding output (index -1)
        all_outputs[-1] = self.base_model.model.embed_tokens(input_ids).detach()
        
        # Layer outputs
        for idx in range(self.num_layers):
            if idx in self.layer_outputs:
                all_outputs[idx] = self.layer_outputs[idx]
            elif idx < len(outputs.hidden_states) - 1:
                all_outputs[idx] = outputs.hidden_states[idx + 1]
        
        # Final logits
        all_outputs['final'] = outputs.logits
        
        return all_outputs

In [ ]:
# Create wrapped model
wrapped_model = ModelWithLayerOutputs(model, NUM_LAYERS)
wrapped_model.eval()

# Test inference
test_input = tokenizer("Привет", return_tensors="pt").to(DEVICE)
with torch.no_grad():
    outputs = wrapped_model(test_input['input_ids'], test_input['attention_mask'])

print(f"Embedding output shape: {outputs[-1].shape}")
for layer_idx in sorted([k for k in outputs.keys() if isinstance(k, int)])[:5]:
    print(f"Layer {layer_idx} output shape: {outputs[layer_idx].shape}")
print(f"...")
print(f"Final logits shape: {outputs['final'].shape}")

In [ ]:
# Save PyTorch model
pt_path = os.path.join(OUTPUT_DIR, 'qwen_layer_model.pt')

torch.save({
    'model_state_dict': wrapped_model.state_dict(),
    'config': config,
    'num_layers': NUM_LAYERS
}, pt_path)

logger.info(f"PyTorch model saved: {pt_path}")

In [ ]:
# Export to ONNX
try:
    onnx_path = os.path.join(OUTPUT_DIR, 'qwen_with_layers.onnx')
    
    seq_len = 32  # Fixed length for export
    dummy_input_ids = torch.randint(0, 1000, (1, seq_len), dtype=torch.long).to(DEVICE)
    dummy_attention_mask = torch.ones(1, seq_len, dtype=torch.long).to(DEVICE)
    
    torch.onnx.export(
        wrapped_model,
        (dummy_input_ids, dummy_attention_mask),
        onnx_path,
        input_names=["input_ids", "attention_mask"],
        output_names=[f"layer_{i}" for i in range(NUM_LAYERS)] + ["logits"],
        dynamic_axes={
            "input_ids": {0: "batch", 1: "seq"},
            "attention_mask": {0: "batch", 1: "seq"},
        },
        opset_version=14,
        do_constant_folding=True
    )
    
    logger.info(f"ONNX exported: {onnx_path}")
    
    # Verify file
    !ls -lh $onnx_path
    
except Exception as e:
    logger.warning(f"ONNX export failed: {e}")
    print("Continuing without ONNX export")

In [ ]:
# Export to OpenVINO (if ONNX export succeeded)
if os.path.exists(onnx_path):
    try:
        import openvino as ov
        
        ov_path = os.path.join(OUTPUT_DIR, 'qwen_with_layers.xml')
        
        ov_model = ov.convert_model(onnx_path)
        ov.serialize(ov_model, ov_path)
        
        logger.info(f"OpenVINO exported: {ov_path}")
        
        # Verify files
        !ls -lh $ov_path
        !ls -lh {ov_path.replace('.xml', '.bin')}
        
    except Exception as e:
        logger.warning(f"OpenVINO export failed: {e}")
        print("OpenVINO export not available")

In [ ]:
# Create inference wrapper script
wrapper_code = '''"""\nLayer-wise inference for EVA-Ai FCP system.\n\nUsage:\n    from qwen_layer_inference import QwenLayerInference\n    \n    model = QwenLayerInference(model_path=\"path/to/model\")\n    model.load()\n    \n    layer_outputs = model.get_layer_outputs(input_ids)\n    for layer_idx, hs in layer_outputs.items():\n        print(f\"Layer {layer_idx}: {hs.shape}\")\n"""\n
import numpy as np\nfrom typing import Dict, Optional, Tuple\nimport logging\n
logger = logging.getLogger(\"qwen_layer_inference\")\n
try:\n    import torch\n    from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer\n    HAS_TORCH = True\nexcept ImportError:\n    HAS_TORCH = False\n

class QwenLayerInference:\n    def __init__(self, model_path: str, device: str = \"cpu\", num_layers: int = 36):\n        self.model_path = model_path\n        self.device = device\n        self.num_layers = num_layers\n        self.model = None\n        self.config = None\n        self.tokenizer = None\n
    def load(self) -> bool:\n        if not HAS_TORCH:\n            logger.error(\"PyTorch not available\")\n            return False\n
        self.config = AutoConfig.from_pretrained(\n            self.model_path,\n            trust_remote_code=True,\n            output_hidden_states=True\n        )\n
        self.tokenizer = AutoTokenizer.from_pretrained(\n            self.model_path,\n            trust_remote_code=True\n        )\n
        self.model = AutoModelForCausalLM.from_pretrained(\n            self.model_path,\n            config=self.config,\n            trust_remote_code=True,\n            torch_dtype=torch.float16 if self.device == \"cuda\" else torch.float32,\n            device_map=self.device,\n            low_cpu_mem_usage=True\n        )\n
        self.model.eval()\n        logger.info(f\"Model loaded on {self.device}\")\n        return True\n
    def get_layer_outputs(self, input_ids, attention_mask=None) -> Dict:\n        if self.model is None:\n            raise RuntimeError(\"Model not loaded\")\n
        with torch.no_grad():\n            outputs = self.model(\n                input_ids=input_ids,\n                attention_mask=attention_mask,\n                output_hidden_states=True\n            )\n
        hidden_states = outputs.hidden_states\n
        result = {}\n        result[-1] = hidden_states[0]  # Embedding\n
        for idx, hs in enumerate(hidden_states[1:-1]):\n            result[idx] = hs\n
        result['final'] = outputs.logits\n
        return result\n
    def forward_layer(self, layer_idx: int, hidden_states, attention_mask=None):\n        if self.model is None:\n            raise RuntimeError(\"Model not loaded\")\n
        layer = self.model.model.layers[layer_idx]\n
        with torch.no_grad():\n            output = layer(hidden_states, attention_mask)\n
        return output[0]\n
    def generate_with_callback(self, prompt: str, max_new_tokens: int = 100, layer_callback=None):\n        if self.model is None:\n            raise RuntimeError(\"Model not loaded\")\n
        inputs = self.tokenizer(prompt, return_tensors=\"pt\")\n        input_ids = inputs[\"input_ids\"].to(self.device)\n        attention_mask = inputs[\"attention_mask\"].to(self.device)\n
        generated_tokens = []\n
        for step in range(max_new_tokens):\n            layer_outputs = self.get_layer_outputs(input_ids, attention_mask)\n
            if layer_callback:\n                for layer_idx in range(self.num_layers):\n                    if layer_idx in layer_outputs:\n                        modified = layer_callback(layer_idx, layer_outputs[layer_idx])\n                        if modified is not None:\n                            layer_outputs[layer_idx] = modified\n
            logits = layer_outputs['final'][0, -1, :]\n            next_token_id = torch.argmax(logits, dim=-1).item()\n
            if next_token_id == self.tokenizer.eos_token_id:\n                break\n
            generated_tokens.append(next_token_id)\n
            input_ids = torch.cat([\n                input_ids,\n                torch.tensor([[next_token_id]], device=self.device)\n            ], dim=-1)\n
            attention_mask = torch.cat([\n                attention_mask,\n                torch.ones((1, 1), device=self.device)\n            ], dim=-1)\n
        return self.tokenizer.decode(generated_tokens, skip_special_tokens=True)\n'''\n
wrapper_path = os.path.join(OUTPUT_DIR, 'qwen_layer_inference.py')
with open(wrapper_path, 'w', encoding='utf-8') as f:\n    f.write(wrapper_code)\n
logger.info(f"Inference wrapper saved: {wrapper_path}")

In [ ]:
# List all exported files
print("\nExported files:")
!ls -lh $OUTPUT_DIR

In [ ]:
# Download files to local (optional)
# from google.colab import files\n# files.download(os.path.join(OUTPUT_DIR, 'qwen_layer_model.pt'))